# F1 pole position predictor

## Import


In [48]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

qualifying = pd.read_csv("F1(1950 - 2024) data\\raw\\qualifying.csv" , na_values=['\\N'])
races = pd.read_csv("F1(1950 - 2024) data\\raw\\races.csv", na_values=['\\N'])
drivers = pd.read_csv("F1(1950 - 2024) data\\raw\\drivers.csv", na_values=['\\N'])
constructors = pd.read_csv("F1(1950 - 2024) data\\raw\\constructors.csv", na_values=['\\N'])

print('Data loaded successfully')

Data loaded successfully


## Checking data

In [49]:
print(qualifying.shape)
qualifying.head()

(10494, 9)


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236


## check null

In [50]:
qualifying.isnull().sum()

qualifyId           0
raceId              0
driverId            0
constructorId       0
number              0
position            0
q1                156
q2               4647
q3               6865
dtype: int64

In [51]:
races.sample(5)

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
1095,1115,2023,17,78,Qatar Grand Prix,2023-10-08,17:00:00,https://en.wikipedia.org/wiki/2023_Qatar_Grand...,2023-10-06,13:30:00,2023-10-07,13:00:00,NaN,NaN,2023-10-06,17:00:00,2023-10-07,17:30:00
333,334,1990,14,26,Spanish Grand Prix,1990-09-30,NaN,http://en.wikipedia.org/wiki/1990_Spanish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,57,2006,5,20,European Grand Prix,2006-05-07,14:00:00,http://en.wikipedia.org/wiki/2006_European_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
306,307,1991,3,21,San Marino Grand Prix,1991-04-28,NaN,http://en.wikipedia.org/wiki/1991_San_Marino_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
547,548,1977,6,6,Monaco Grand Prix,1977-05-22,NaN,http://en.wikipedia.org/wiki/1977_Monaco_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data manipulation

In [52]:
df = pd.merge(qualifying, races, how='left', on='raceId')
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
4081,4083,90,31,3,3,3,1:24.998,NaN,NaN,2004,1,1,Australian Grand Prix,2004-03-07,NaN,http://en.wikipedia.org/wiki/2004_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1782,1783,201,64,21,16,18,1:45.588,NaN,NaN,1998,11,10,German Grand Prix,1998-08-02,NaN,http://en.wikipedia.org/wiki/1998_German_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1306,1307,85,21,4,6,9,1:22.068,NaN,NaN,2005,15,14,Italian Grand Prix,2005-09-04,14:00:00,http://en.wikipedia.org/wiki/2005_Italian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9301,9359,1082,839,214,31,7,1:33.012,1:26.135,1:23.529,2022,9,7,Canadian Grand Prix,2022-06-19,18:00:00,http://en.wikipedia.org/wiki/2022_Canadian_Gra...,2022-06-17,18:00:00,2022-06-17,21:00:00,2022-06-18,17:00:00,2022-06-18,20:00:00,NaN,NaN
8393,8417,1032,154,210,8,20,NaN,NaN,NaN,2020,2,70,Styrian Grand Prix,2020-07-12,13:10:00,http://en.wikipedia.org/wiki/2020_Styrian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Drop pre 2005

In [53]:
df = df[df['year'] >= 2005]

## Convert Str to seconds

In [54]:
df['q1'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [55]:
df['q1'] = df['q1'].str.split(':').str[0].astype(float) * 60 + df['q1'].str.split(':').str[1].astype(float)

df['q2'] = df['q2'].str.split(':').str[0].astype(float) * 60 + df['q2'].str.split(':').str[1].astype(float)

df['q3'] = df['q3'].str.split(':').str[0].astype(float) * 60 + df['q3'].str.split(':').str[1].astype(float)


In [56]:
df['q1'].head(10)

0    86.572
1    86.103
2    85.664
3    85.994
4    85.960
5    86.427
6    86.295
7    86.381
8    86.919
9    86.702
Name: q1, dtype: float64

## Filling nulls with per year medians

In [57]:
df['q1'] = df.groupby('year')['q1'].transform(lambda x: x.fillna(x.median()))
df['q2'] = df.groupby('year')['q2'].transform(lambda x: x.fillna(x.median()))
df['q3'] = df.groupby('year')['q3'].transform(lambda x: x.fillna(x.median()))


In [58]:
df[['q1', 'q2', 'q3']].isnull().sum()

q1      0
q2      0
q3    376
dtype: int64

## Filling q3 nulls with 0

In [59]:
df['q3'] = df['q3'].fillna(0)

## create reached_q2&q3 column

In [60]:
df['reached_q2'] = (df['q2'].notnull()).astype(int)
df['reached_q3'] = (df['q3'].notnull()).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3
7142,7165,970,13,3,19,6,94.205,93.759,93.5070,2017,2,17,Chinese Grand Prix,2017-04-09,06:00:00,http://en.wikipedia.org/wiki/2017_Chinese_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
8949,8990,1064,20,117,5,17,70.731,81.086,81.0545,2021,13,39,Dutch Grand Prix,2021-09-05,13:00:00,http://en.wikipedia.org/wiki/2021_Dutch_Grand_...,2021-09-03,NaN,2021-09-03,NaN,2021-09-04,NaN,2021-09-04,NaN,NaN,NaN,1,1
7433,7456,984,842,5,10,17,91.317,90.307,89.4800,2017,16,22,Japanese Grand Prix,2017-10-08,05:00:00,http://en.wikipedia.org/wiki/2017_Japanese_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
3491,3493,343,812,164,20,24,92.060,91.155,90.7050,2010,7,5,Turkish Grand Prix,2010-05-30,11:00:00,http://en.wikipedia.org/wiki/2010_Turkish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
6637,6660,944,834,206,53,18,76.151,93.361,89.4800,2015,18,18,Brazilian Grand Prix,2015-11-15,16:00:00,http://en.wikipedia.org/wiki/2015_Brazilian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1


In [61]:
print(df[df['q3'].isnull()]['reached_q3'].value_counts())

Series([], Name: count, dtype: int64)


## create got_pole column

In [62]:
df['got_pole'] = (df['position'] == 1).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
9586,9644,1098,822,51,77,12,91.504,91.443,84.9105,2023,1,3,Bahrain Grand Prix,2023-03-05,15:00:00,https://en.wikipedia.org/wiki/2023_Bahrain_Gra...,2023-03-03,11:30:00,2023-03-03,15:00:00,2023-03-04,11:30:00,2023-03-04,15:00:00,NaN,NaN,1,1,0
7139,7162,970,822,131,77,3,93.684,92.552,91.8650,2017,2,17,Chinese Grand Prix,2017-04-09,06:00:00,http://en.wikipedia.org/wiki/2017_Chinese_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
5129,5132,866,39,164,23,24,78.330,95.725,95.6570,2012,7,7,Canadian Grand Prix,2012-06-10,18:00:00,http://en.wikipedia.org/wiki/2012_Canadian_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
3203,3205,11,69,6,3,20,101.413,91.222,92.3950,2009,11,12,European Grand Prix,2009-08-23,12:00:00,http://en.wikipedia.org/wiki/2009_European_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
9723,9781,1106,857,1,81,9,82.190,79.659,91.3490,2023,8,7,Canadian Grand Prix,2023-06-18,18:00:00,https://en.wikipedia.org/wiki/2023_Canadian_Gr...,2023-06-16,17:30:00,2023-06-16,21:00:00,2023-06-17,16:30:00,2023-06-17,20:00:00,NaN,NaN,1,1,0


In [63]:
df['got_pole'].value_counts()

got_pole
0    7872
1     394
Name: count, dtype: int64

In [64]:
df.groupby('year')['q2'].median().head(20)

year
2005    89.1590
2006    81.4840
2007    83.1205
2008    83.4555
2009    91.2220
         ...   
2020    79.9620
2021    81.0860
2022    85.2250
2023    84.4405
2024    83.8770
Name: q2, Length: 20, dtype: float64

In [65]:
for i, col in enumerate(df.columns):
    print(i, col)

pd.set_option('display.max_columns', None)
df.sample(20)

0 qualifyId
1 raceId
2 driverId
3 constructorId
4 number
5 position
6 q1
7 q2
8 q3
9 year
10 round
11 circuitId
12 name
13 date
14 time
15 url
16 fp1_date
17 fp1_time
18 fp2_date
19 fp2_time
20 fp3_date
21 fp3_time
22 quali_date
23 quali_time
24 sprint_date
25 sprint_time
26 reached_q2
27 reached_q3
28 got_pole


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
4432,4434,71,33,17,18,14,106.846,93.4830,0.0000,2005,1,1,Australian Grand Prix,2005-03-06,14:00:00,http://en.wikipedia.org/wiki/2005_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
251,252,30,8,6,1,4,106.960,106.2980,107.9920,2008,13,13,Belgian Grand Prix,2008-09-07,12:00:00,http://en.wikipedia.org/wiki/2008_Belgian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
8272,8296,1026,825,210,20,19,86.273,86.1605,85.7870,2019,17,22,Japanese Grand Prix,2019-10-13,05:10:00,http://en.wikipedia.org/wiki/2019_Japanese_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
4544,4546,841,67,5,18,10,86.232,85.8820,87.0660,2011,1,1,Australian Grand Prix,2011-03-27,06:00:00,http://en.wikipedia.org/wiki/2011_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
8017,8041,1014,830,9,33,4,77.244,76.7260,76.3570,2019,5,4,Spanish Grand Prix,2019-05-12,13:10:00,http://en.wikipedia.org/wiki/2019_Spanish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10295,10353,1135,830,9,1,2,71.393,70.8110,70.0290,2024,15,39,Dutch Grand Prix,2024-08-25,13:00:00,https://en.wikipedia.org/wiki/2024_Dutch_Grand...,2024-08-23,10:30:00,2024-08-23,14:00:00,2024-08-24,09:30:00,2024-08-24,13:00:00,NaN,NaN,1,1,0
12,13,18,18,11,16,13,86.712,86.2590,86.7915,2008,1,1,Australian Grand Prix,2008-03-16,04:30:00,http://en.wikipedia.org/wiki/2008_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
3575,3577,347,155,15,23,12,75.951,75.0840,90.7050,2010,11,10,German Grand Prix,2010-07-25,12:00:00,http://en.wikipedia.org/wiki/2010_German_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
8546,8570,1040,840,211,18,13,93.852,93.3640,81.0110,2020,10,71,Russian Grand Prix,2020-09-27,11:10:00,http://en.wikipedia.org/wiki/2020_Russian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0


## Features and Target

In [66]:
features = ['q1', 'q2', 'q3', 'reached_q2', 'reached_q3', 'year', 'circuitId', 'constructorId', 'driverId', 'round']
target = 'got_pole'

## New df

In [67]:
df_cleaned = df[features + [target]]
print(df_cleaned.shape)
df_cleaned.sample(5)

(8266, 11)


,q1,q2,q3,reached_q2,reached_q3,year,circuitId,constructorId,driverId,round,got_pole
4616,87.013,86.395,90.3075,1,1,2011,5,6,13,4,0
12,86.712,86.259,86.7915,1,1,2008,1,11,18,1,0
6987,94.460,93.609,93.2640,1,1,2016,2,131,3,16,0
5714,105.657,104.588,90.9080,1,1,2013,15,5,818,13,0
9928,96.009,95.880,84.9105,1,1,2023,69,210,825,18,0


In [69]:
print(df_cleaned.shape)
print(df_cleaned['got_pole'].value_counts())
print(df_cleaned.isnull().sum())

(8266, 11)
got_pole
0    7872
1     394
Name: count, dtype: int64
q1               0
q2               0
q3               0
reached_q2       0
reached_q3       0
                ..
circuitId        0
constructorId    0
driverId         0
round            0
got_pole         0
Length: 11, dtype: int64
